# QLoRA fine-tune — Qwen2.5-3B-Instruct incident reports

Thin runner. **All training logic lives in `src/train_qlora.py`** — this notebook
installs dependencies, locates the data, and calls it. Do not paste training code
here; two copies will drift.

Requires GPU + Internet in session settings (set via `kernel-metadata.json` when
pushed from the CLI).

In [ ]:
# 1. Dependencies. Kaggle images ship torch + CUDA; these are the additions.
!pip install -q -U "transformers>=4.44,<6" "peft>=0.12" "bitsandbytes>=0.43" \
    "accelerate>=0.33" "datasets>=2.20" sentencepiece

In [ ]:
# 2. Locate the input dataset and the project source.
#    Kaggle's mount layout varies (/kaggle/input/<slug> on standard images,
#    /kaggle/input/datasets/<owner>/<slug> on the private BYOD image). Discover
#    the data by finding train.jsonl rather than hardcoding a path.
import os, shutil
from pathlib import Path

ROOT = Path("/kaggle/input")
print("input tree:")
for p in sorted(ROOT.rglob("*"))[:40]:
    print("  ", p)

matches = sorted(ROOT.rglob("train.jsonl"))
assert matches, f"train.jsonl not found anywhere under {ROOT}"
INPUT = matches[0].parent
print("\nresolved dataset dir:", INPUT)

WORK = Path("/kaggle/working")

src_dirs = sorted(ROOT.rglob("train_qlora.py"))
assert src_dirs, f"train_qlora.py not found under {ROOT}"
SRC = src_dirs[0].parent
print("resolved src dir    :", SRC)

# Copy src/ into the working dir so `python -m src.train_qlora` resolves, and so
# the read-only input mount is never written to.
shutil.copytree(SRC, WORK / "src", dirs_exist_ok=True)

DATA = WORK / "data" / "processed"
DATA.mkdir(parents=True, exist_ok=True)
for name in ("train.jsonl", "test.jsonl"):
    shutil.copy(INPUT / name, DATA / name)

os.chdir(WORK)
print("\ncwd:", Path.cwd())
print("train:", sum(1 for _ in open(DATA / "train.jsonl")), "examples")
print("test :", sum(1 for _ in open(DATA / "test.jsonl")), "examples")

In [ ]:
# 3. Confirm the GPU before spending an hour on it.
import torch
print("cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU in session settings."
print("device:", torch.cuda.get_device_name(0))
print("memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# 4. Smoke test first -- tiny randomly-initialised stand-in, 2 steps. Proves the
#    data path and the loss masking before committing to the full run.
!python -m src.train_qlora --smoke --train data/processed/train.jsonl

In [ ]:
# 5. The real run. Hyperparameters are constants at the top of src/train_qlora.py.
!python -m src.train_qlora \
    --train data/processed/train.jsonl \
    --out /kaggle/working/qlora-adapter

In [ ]:
# 6. Fail loudly if training produced nothing.
#    A previous version of this notebook was skipped by nbconvert and still
#    reported COMPLETE; this cell makes that impossible to miss.
import json
from pathlib import Path

OUT = Path("/kaggle/working/qlora-adapter")
weights = list(OUT.glob("adapter_model.safetensors")) + list(OUT.glob("adapter_model.bin"))
assert weights, f"NO ADAPTER WEIGHTS in {OUT} -- training did not run. Contents: {list(OUT.glob('*'))}"

summary = json.loads((OUT / "training_summary.json").read_text())
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(OUT)}  {p.stat().st_size / 1e6:.2f} MB")

print()
print("steps           :", summary["steps"])
print("elapsed (min)   :", round(summary["elapsed_seconds"] / 60, 1))
print("final train loss:", summary["final_train_loss"])
print("final eval loss :", summary["final_eval_loss"])
print("gpu             :", summary["gpu"])
print()
print("per-epoch history:")
for h in summary["log_history"]:
    if "eval_loss" in h:
        print(f"  epoch {h.get('epoch'):.2f}  eval_loss {h['eval_loss']:.4f}")

In [ ]:
# 7. Predictions for both variants, so the SCOPE.md 5.3 comparison uses one harness.
!python -m src.generate_predictions --model base  --load-4bit \
    --test data/processed/test.jsonl
!python -m src.generate_predictions --model tuned --load-4bit \
    --adapter /kaggle/working/qlora-adapter \
    --test data/processed/test.jsonl

In [ ]:
# 8. Move predictions to /kaggle/working root so they appear in kernel output.
import shutil
from pathlib import Path

for name in ("preds_base.jsonl", "preds_tuned.jsonl"):
    src = Path("/kaggle/working/data/processed") / name
    if src.exists():
        shutil.copy(src, Path("/kaggle/working") / name)
        print("staged", name, src.stat().st_size, "bytes")
    else:
        print("MISSING", name)

# src/ is input code, not output -- drop it so it is not published as kernel output.
shutil.rmtree("/kaggle/working/src", ignore_errors=True)
shutil.rmtree("/kaggle/working/data", ignore_errors=True)
print("\nfinal /kaggle/working:")
for p in sorted(Path("/kaggle/working").iterdir()):
    print("  ", p.name)